# 10 · Segmentação com rede: U-Net e Dice

Continuação do notebook 08. Lá o coração foi recortado por métodos clássicos (caixa fixa, limiar e
morfologia, bordas do ICT) e o melhor deles chegou a Dice 0,64 contra o contorno à mão. Aqui o recorte é
feito por **redes de segmentação**, que dão uma probabilidade por pixel, como no exemplo de RM cerebral
com U-Net e Dice.

1. **uma rede pré-treinada, sem treino nenhum aqui**: a PSPNet do `torchxrayvision`, treinada em
   contornos de radiologistas, e o pós-processamento pelo maior componente conexo;
2. **uma U-Net pequena treinada aqui**, em Keras, com perda Dice, e uma pergunta sobre o rótulo: o
   que ela aprende com **pseudo-rótulos** de um método clássico (as bordas do ICT) e com os de uma rede
   (a PSPNet)?
3. **duas réguas contra o gabarito**: área (Dice, IoU) e **borda** (distância média e Hausdorff 95, em
   pixels), porque "o pixel da borda pesa mais";
4. **onde acerta e onde erra**: contorno verde do gabarito e vermelho da rede, com o Dice de cada exame;
5. **do recorte ao diagnóstico**: o ICT medido pelo coração recortado, no teste inteiro, contra o ICT
   do perfil (AUC 0,739, notebook 04);
6. **o aviso de método**: de onde vêm os pesos e o que isso faz com o Dice.

O gabarito é o mesmo do notebook 08: 29 contornos de um anotador, que não é radiologista.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import warnings

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.model import avaliacao, cnn, pretreinada, unet
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import ict, segmentacao
from src.visao.metricas import dice, distancia_media_borda, hausdorff95, iou

configurar_estilo()
warnings.filterwarnings("ignore", message=".*normaliz.*")
SECAO = "10-segmentacao-unet"
PARAMS_ICT = config.PARAMS_ICT  # escolhidos no notebook 04
# True refaz o treino das U-Nets; False reaproveita models/unet_*.keras se existirem.
RETREINAR = False
keras.utils.set_random_seed(config.SEMENTE)

baixar_base()
catalogo = dividir(listar_imagens())
imagens = carregar_imagens(catalogo["caminho"])
y = catalogo["classe"].to_numpy()
conjunto = catalogo["conjunto"].to_numpy()
ids = catalogo["id"].to_numpy()
pos = {i: k for k, i in enumerate(ids)}

marcacoes = segmentacao.carregar_marcacoes()
primeiro = sorted(marcacoes["anotador"].unique())[0]
gabarito = {r["id"]: segmentacao.poligono_para_mascara(r["pontos"])
            for _, r in marcacoes[marcacoes["anotador"] == primeiro].iterrows()}
ids_gab = list(gabarito)
print(f"{len(imagens)} raio-X · {len(gabarito)} contornos à mão")

medidas = [ict.medir_ict(im, **PARAMS_ICT) for im in imagens]
medidas_treino = [m for m, c in zip(medidas, conjunto) if c == "treino"]
CAIXA, _ = ict.caixas_de_referencia(medidas_treino)

def contorno_da_mascara(ax, mascara, cor, lw=1.4):
    if mascara.any():
        ax.contour(mascara.astype(float), levels=[0.5], colors=[cor], linewidths=lw)

## 1. Uma rede pré-treinada, sem treino aqui

A PSPNet do `torchxrayvision` segmenta 14 estruturas do tórax; usamos **coração** e os **dois pulmões**.
Ela espera 512 × 512 e as nossas imagens têm 128 × 128: a imagem é ampliada, a saída é reduzida de
volta. Nada nela foi ajustado para esta base. Roda uma vez nos 5.552 exames e fica em cache.

O **pós-processamento** é o do exemplo de RM: fica só o **maior componente conexo**, porque o coração é
um só e ilhas soltas são erro.

In [ ]:
prob = pretreinada.segmentar_com_cache(ids, imagens)
p_coracao = prob[:, 0].astype(np.float32)
pulmoes = (prob[:, 1] > 0.5) | (prob[:, 2] > 0.5)

psp_cru = p_coracao > 0.5
psp = np.stack([segmentacao.maior_componente(m) for m in psp_cru])
n_ilhas = np.array([(m & ~c).sum() > 0 for m, c in zip(psp_cru, psp)])
print(f"Exames em que o maior componente tirou alguma ilha: {n_ilhas.mean():.1%}")
print(f"Exames sem coração nenhum: {(~psp.any(axis=(1, 2))).sum()}")

d_cru = [dice(gabarito[i], psp_cru[pos[i]]) for i in ids_gab]
d_pos = [dice(gabarito[i], psp[pos[i]]) for i in ids_gab]
print(f"Dice contra o gabarito: sem pós-processamento {np.mean(d_cru):.3f} · com maior componente {np.mean(d_pos):.3f}")

fig, eixos = plt.subplots(1, 4, figsize=(13, 3.6))
k = pos[ids_gab[10]]
eixos[0].imshow(imagens[k], cmap="gray"); eixos[0].set_title("raio-X 128 × 128", fontsize=9)
for ax, (nome, mapa) in zip(eixos[1:], [("coração", prob[k, 0]), ("pulmão esquerdo", prob[k, 1]), ("pulmão direito", prob[k, 2])]):
    ax.imshow(imagens[k], cmap="gray")
    ax.imshow(mapa.astype(float), cmap="magma", alpha=0.55, vmin=0, vmax=1)
    ax.set_title(f"probabilidade · {nome}", fontsize=9)
for ax in eixos:
    ax.axis("off")
salvar_figura(fig, SECAO, "pretreinada-mapas")
plt.show()

## 2. Uma U-Net treinada aqui: o aluno aprende o que o professor ensina

Não temos contornos à mão para 3.550 raio-X de treino, só para 29 do teste. O treino usa
**pseudo-rótulos**, como o YOLO do notebook 09, de dois professores diferentes:

* **bordas do ICT** (notebook 08): o melhor método clássico, Dice 0,64 contra o gabarito;
* **PSPNet** (seção 1): uma rede que já viu contornos de radiologista.

A mesma U-Net (4 níveis, 16 a 128 filtros, ≈ 480 mil pesos), a mesma perda (entropia cruzada + 1 − Dice),
treino e validação da divisão do projeto, parada antecipada pelo Dice de validação. O teste fica de
fora. Exames sem medida do ICT saem do treino do primeiro professor.

In [ ]:
X = cnn.tensores(imagens)
professores = {
    "bordas do ICT": np.stack([segmentacao.mascara_pelo_ict(m) for m in medidas]),
    "PSPNet": psp,
}
tr, va = conjunto == "treino", conjunto == "validacao"
alunos, historicos = {}, {}
for nome, rotulo in professores.items():
    arquivo = config.MODELS / f"unet_{'ict' if 'ICT' in nome else 'pspnet'}.keras"
    alunos[nome], h = unet.treinar_ou_carregar(arquivo, X, rotulo, tr, va, retreinar=RETREINAR)
    historicos[nome] = pd.DataFrame(h)
print("Pesos da U-Net:", f"{alunos['PSPNet'].count_params():,}".replace(",", "."))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
curvas = []
for cor, (nome, h) in zip([config.AZUL_CLARO, config.CORAL], historicos.items()):
    ax.plot(h.index + 1, h["dice"], color=cor, ls="--", label=f"{nome} · treino")
    ax.plot(h.index + 1, h["val_dice"], color=cor, lw=2, label=f"{nome} · validação")
    curvas.append(h.assign(professor=nome, epoca=h.index + 1))
ax.set(xlabel="época", ylabel="Dice com o próprio professor", ylim=(0.5, 1), title="Treino das U-Nets")
ax.legend(fontsize=8)
salvar_figura(fig, SECAO, "curvas-treino", pd.concat(curvas))
plt.show()

mascaras_unet = {nome: np.stack([segmentacao.maior_componente(p > 0.5) for p in unet.prever(m, X)])
                 for nome, m in alunos.items()}
te = conjunto == "teste"
for nome in alunos:
    d = [dice(a, b) for a, b in zip(mascaras_unet[nome][te], professores[nome][te])]
    print(f"U-Net ({nome}) reproduz o professor no teste: Dice médio {np.mean(d):.3f}")

## 3. Duas réguas contra o gabarito: área e borda

Os seis recortes nos 29 exames com contorno à mão:

* **área**: Dice e IoU, com intervalo de 95 % por bootstrap dos exames;
* **borda**: a **distância média** entre os contornos (erro típico, em pixels) e o **Hausdorff 95**
  (o pedaço que mais escapou, sem ser refém de um pixel).

Num coração de 40 pixels de largura, 3 pixels de borda a mais de cada lado quase não mexem no Dice, mas
mudam o ICT em 0,05: a régua de borda é a que importa para medir.

In [ ]:
metodos = {
    "caixa fixa": lambda k: segmentacao.caixa_para_mascara(CAIXA),
    "limiar e morfologia": lambda k: segmentacao.mascara_por_limiar(imagens[k]),
    "bordas do ICT": lambda k: professores["bordas do ICT"][k],
    "PSPNet (pré-treinada)": lambda k: psp[k],
    "U-Net · professor ICT": lambda k: mascaras_unet["bordas do ICT"][k],
    "U-Net · professor PSPNet": lambda k: mascaras_unet["PSPNet"][k],
}
CORES = dict(zip(metodos, [config.AMARELO, config.VERDE, config.AZUL_CLARO, config.ROXO, config.AZUL_COBALTO, config.CORAL]))

linhas = []
for i in ids_gab:
    k, gab = pos[i], gabarito[i]
    for nome, f in metodos.items():
        m = f(k)
        linhas.append({"id": i, "classe": y[k], "metodo": nome, "dice": dice(gab, m), "iou": iou(gab, m),
                       "borda_media_px": distancia_media_borda(gab, m), "hausdorff95_px": hausdorff95(gab, m),
                       "area_vs_gabarito": m.sum() / gab.sum()})
comparacao = pd.DataFrame(linhas)
salvar_tabela(comparacao, SECAO, "metricas-por-exame", indice=False)

rng = np.random.default_rng(config.SEMENTE)
def ic(v, n=2000):
    v = np.asarray(v)
    return np.percentile([rng.choice(v, len(v)).mean() for _ in range(n)], [2.5, 97.5])

resumo = []
for nome, g in comparacao.groupby("metodo", sort=False):
    b, a = ic(g["dice"])
    resumo.append({"metodo": nome, "dice_medio": g["dice"].mean(), "ic_baixo": b, "ic_alto": a,
                   "dice_mediano": g["dice"].median(), "iou_medio": g["iou"].mean(),
                   "borda_media_px": g["borda_media_px"].median(), "hausdorff95_px": g["hausdorff95_px"].median(),
                   "area_vs_gabarito": g["area_vs_gabarito"].median()})
resumo = pd.DataFrame(resumo).set_index("metodo")
salvar_tabela(resumo, SECAO, "metricas-resumo")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for j, (nome, g) in enumerate(comparacao.groupby("metodo", sort=False)):
    jit = j + rng.uniform(-0.13, 0.13, len(g))
    ax1.scatter(g["dice"], jit, color=CORES[nome], s=20, alpha=0.75)
    ax1.errorbar(resumo.loc[nome, "dice_medio"], j, xerr=[[resumo.loc[nome, "dice_medio"] - resumo.loc[nome, "ic_baixo"]],
                 [resumo.loc[nome, "ic_alto"] - resumo.loc[nome, "dice_medio"]]], fmt="D", color="#333", ms=6, capsize=4)
    ax2.scatter(g["hausdorff95_px"], jit, color=CORES[nome], s=20, alpha=0.75)
    ax2.plot(g["hausdorff95_px"].median(), j, "D", color="#333", ms=6)
ax1.set(yticks=range(len(metodos)), yticklabels=list(metodos), xlim=(0, 1), xlabel="Dice (maior = melhor)",
        title="Área: Dice (média e IC 95 %)")
ax2.set(xlabel="Hausdorff 95 em pixels (menor = melhor)", title="Borda: Hausdorff 95 (mediana)")
salvar_figura(fig, SECAO, "area-e-borda")
plt.show()
resumo.round(3)

In [ ]:
# A U-Net que aprendeu com a PSPNet contra as bordas do ICT e contra a própria PSPNet: diferença pareada.
pareado = comparacao.pivot(index="id", columns="metodo", values="dice")
for a, b in [("U-Net · professor PSPNet", "bordas do ICT"), ("U-Net · professor PSPNet", "PSPNet (pré-treinada)"),
             ("U-Net · professor ICT", "bordas do ICT")]:
    dif = (pareado[a] - pareado[b]).to_numpy()
    lo, hi = ic(dif)
    print(f"Dice {a} − {b}: {dif.mean():+.3f} (IC 95 % {lo:+.3f} a {hi:+.3f})")

## 4. Onde acerta e onde erra

A convenção do exemplo de RM: **verde** é o gabarito, **vermelho** é a rede (U-Net com professor
PSPNet), e o Dice de cada exame no título. Primeiro os seis melhores, depois os seis piores.

In [ ]:
escolhido = "U-Net · professor PSPNet"
g = comparacao[comparacao["metodo"] == escolhido].sort_values("dice", ascending=False)
casos = pd.concat([g.head(6), g.tail(6)])
fig, eixos = plt.subplots(2, 6, figsize=(15, 5.8))
for ax, (_, r) in zip(eixos.ravel(), casos.iterrows()):
    k = pos[r["id"]]
    ax.imshow(imagens[k], cmap="gray")
    contorno_da_mascara(ax, gabarito[r["id"]], "#2ECC71")
    contorno_da_mascara(ax, mascaras_unet["PSPNet"][k], "#E74C3C")
    ax.set_title(f"{r['id']} · {config.ROTULOS[r['classe']]}\nDice {r['dice']:.2f} · HD95 {r['hausdorff95_px']:.0f} px", fontsize=7)
    ax.axis("off")
eixos[0, 0].text(-6, 64, "melhores", rotation=90, va="center", ha="right", fontsize=10, fontweight="bold", color=config.AZUL_PROFUNDO)
eixos[1, 0].text(-6, 64, "piores", rotation=90, va="center", ha="right", fontsize=10, fontweight="bold", color=config.AZUL_PROFUNDO)
fig.suptitle("U-Net (professor PSPNet) · verde = contorno à mão, vermelho = rede", fontweight="bold", color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "melhores-e-piores", casos)
plt.show()

fig, eixos = plt.subplots(1, 6, figsize=(15, 3))
k = pos[g.iloc[len(g) // 2]["id"]]
for ax, nome in zip(eixos, metodos):
    ax.imshow(imagens[k], cmap="gray")
    contorno_da_mascara(ax, gabarito[ids[k]], "#2ECC71")
    contorno_da_mascara(ax, metodos[nome](k), "#E74C3C")
    ax.set_title(f"{nome}\nDice {dice(gabarito[ids[k]], metodos[nome](k)):.2f}", fontsize=8)
    ax.axis("off")
salvar_figura(fig, SECAO, "seis-metodos-um-exame")
plt.show()

## 5. Do recorte ao diagnóstico: o ICT do coração recortado

Com uma máscara do coração em **todos** os 1.114 exames de teste, o ICT sai direto: a maior largura da
máscara dividida pela largura do tórax. Duas versões do denominador:

* **tórax pelo perfil** (notebook 04), o mesmo para todos os métodos: só o coração muda;
* **tórax pelos pulmões da PSPNet**: a largura de ponta a ponta dos dois pulmões. É a medida inteira
  feita por segmentação, sem o perfil.

A comparação com o ICT do perfil é **pareada** (os mesmos exames na mesma reamostragem).

In [ ]:
torax_perfil = np.array([m.torax_px if m.ok else np.nan for m in medidas])
torax_pulmoes = np.array([segmentacao.largura_maxima(p) for p in pulmoes], float)
torax_pulmoes[torax_pulmoes == 0] = np.nan
ict_perfil = np.array([m.ict if m.ok else np.nan for m in medidas])

def largura(mascaras):
    w = np.array([segmentacao.largura_maxima(m) for m in mascaras], float)
    w[w == 0] = np.nan
    return w

escores = {
    "ICT do perfil (notebook 04)": ict_perfil,
    "PSPNet · coração / tórax do perfil": largura(psp) / torax_perfil,
    "PSPNet · coração / pulmões": largura(psp) / torax_pulmoes,
    "U-Net (PSPNet) · coração / tórax do perfil": largura(mascaras_unet["PSPNet"]) / torax_perfil,
    "U-Net (PSPNet) · coração / pulmões": largura(mascaras_unet["PSPNet"]) / torax_pulmoes,
    "U-Net (ICT) · coração / tórax do perfil": largura(mascaras_unet["bordas do ICT"]) / torax_perfil,
}
linhas = []
for nome, e in escores.items():
    a = avaliacao.auc_bootstrap(y[te], e[te])
    c = avaliacao.comparar_auc(y[te], e[te], ict_perfil[te]) if nome != "ICT do perfil (notebook 04)" else {}
    linhas.append({"escore": nome, "n": int(np.isfinite(e[te]).sum()), **a,
                   "dif_vs_perfil": c.get("diferenca", np.nan), "dif_ic_baixo": c.get("ic_baixo", np.nan),
                   "dif_ic_alto": c.get("ic_alto", np.nan), "p_valor": c.get("p_valor", np.nan)})
aucs = pd.DataFrame(linhas).set_index("escore")
salvar_tabela(aucs, SECAO, "ict-da-segmentacao")
aucs.round(3)

In [ ]:
melhor = aucs["auc"].drop("ICT do perfil (notebook 04)").idxmax()
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
for ax, nome in zip(eixos, ["ICT do perfil (notebook 04)", melhor]):
    e = escores[nome][te]
    for c in (0, 1):
        v = e[(y[te] == c) & np.isfinite(e)]
        ax.hist(v, bins=np.linspace(0.25, 0.85, 40), alpha=0.6, color=config.CORES_CLASSE[c], label=config.ROTULOS[c])
    ax.axvline(0.5, color="#555", ls="--", lw=1)
    ax.set(xlabel="ICT", ylabel="exames de teste", title=f"{nome}\nAUC {aucs.loc[nome, 'auc']:.3f}")
    ax.legend()
salvar_figura(fig, SECAO, "ict-por-classe", pd.DataFrame({k: v[te] for k, v in escores.items()}).assign(classe=y[te]))
plt.show()

## 6. O aviso de método: de onde vêm os pesos

A PSPNet foi treinada no **ChestX-Det**, 3.578 raio-X do mesmo ChestX-ray14 do NIH de onde sai a nossa
base, com contornos de radiologistas. Os nomes dos nossos arquivos (`<n>.png`) não permitem saber quais
exames estão nos dois, então **alguns raio-X do nosso teste podem ter sido vistos no treino dela**. É o
mesmo aviso do exemplo de RM cerebral, cujos pesos públicos foram treinados no próprio conjunto
avaliado.

O que isso afeta e o que não afeta:

* o **Dice da PSPNet** e, por herança, o da U-Net que aprendeu com ela podem estar **otimistas**;
* a **U-Net** em si só viu o **treino** da nossa divisão: o que ela aprendeu do teste veio, no máximo,
  pela mão do professor;
* a **AUC do ICT** não usa rótulos de cardiomegalia em lugar nenhum da segmentação (a PSPNet aprendeu
  anatomia, não doença), então o vazamento de pixels não vira vazamento de diagnóstico.

## Conclusões

* **As redes recortam melhor que os métodos clássicos, e a diferença aparece mais na borda que na
  área.** A U-Net com professor PSPNet chega a Dice médio **0,69** (IC 95 % 0,61–0,76; mediana 0,75), a
  PSPNet pré-treinada a 0,69 e as bordas do ICT a 0,64. Na borda o ganho é mais claro: Hausdorff 95
  mediano de **11 px** nas redes contra 16 px das bordas do ICT e 24–25 px do limiar e da caixa. A
  distância média do contorno cai de 6,2 para 4,2 px.
* **O aluno vira o professor, nem mais nem menos.** Cada U-Net reproduz o próprio professor no teste
  com Dice 0,87, e contra o gabarito empata com ele: +0,007 (IC −0,013 a +0,027) sobre as bordas do
  ICT e +0,005 (IC −0,013 a +0,025) sobre a PSPNet. É o mesmo resultado do YOLO no notebook 09: o que
  limita a rede é a qualidade do rótulo, não a arquitetura. Com o mesmo treino, trocar o professor
  clássico pela PSPNet vale +0,05 de Dice (IC −0,02 a +0,12, n = 29).
* **O aluno corrige uma falha do professor.** A PSPNet não acha coração nenhum em 127 dos 5.552
  exames (2,3 %); a U-Net que aprendeu com ela sempre devolve um recorte. O maior componente conexo
  quase não muda nada (tira ilhas em 2 % dos exames, Dice igual).
* **Os piores casos são do gabarito.** Nos exames 1 a 4, os primeiros marcados, todos os métodos
  caem (U-Net 0,53 contra 0,72 nos demais). Em `test/false/200` e `test/true/376` o contorno à mão
  abraça só o mediastino alto: a rede está mais perto do coração que o gabarito.
* **Do recorte ao diagnóstico: o melhor ICT do projeto.** A largura do coração da U-Net dividida pela
  largura dos pulmões da PSPNet dá **AUC 0,803** (IC 0,776–0,830) no teste inteiro, **+0,064**
  (IC +0,025 a +0,100) sobre o ICT do perfil, comparação pareada. É uma medida interpretável (um ICT de
  verdade) que empata com a CNN do notebook 05 (0,802), sem nunca ter visto um rótulo de
  cardiomegalia.
* **A régua tem de ser coerente.** O mesmo coração dividido pela largura do tórax do perfil piora
  (AUC 0,69–0,71): o perfil mede o tórax nas linhas 50 a 90, e o coração recortado tem outra altura.
  Coração e tórax precisam vir do mesmo tipo de medida.
* **O limite está no aviso da seção 6.** A PSPNet pode ter visto parte do nosso teste no treino dela,
  e o gabarito é de uma pessoa só, que não é radiologista, em 29 exames. Os Dice daqui servem para
  ordenar os métodos; não são o Dice que um radiologista mediria.